<a href="https://colab.research.google.com/github/Maybeoff/google-colab/blob/main/ipynb/clear_ComfyUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Ultimate ComfyUI Colab Template

Welcome to this ready-to-use Google Colab notebook for running the **latest version of ComfyUI**! This template provides a fully automated, hassle-free setup to get you generating images in the cloud in no time.

## ✨ Features
- **Always Up-to-Date:** Automatically fetches and installs the most recent version of ComfyUI directly from the source.
- **Multiple Access Methods:** Choose how you want to expose the web interface. We support several tunneling options so you always have a working connection:
  - `localtunnel` (Quick and easy)
  - `tuna.am` (A very stable service for Russian and CIS users. Requires authorization.)
  - `cloudflare` (Fast and reliable)
- **Pre-configured Environment:** All necessary dependencies, xformers, and essential custom nodes are handled for you.

## 🛠️ How to Use This Notebook

1. **Enable GPU:** Ensure your Colab runtime is utilizing a GPU. Go to `Runtime` > `Change runtime type` > `Hardware accelerator` and select `T4 GPU` (or better).
2. **Setup Environment:** Run the setup cell below to install ComfyUI and its requirements.
3. **Download Models:** Use the provided cells to fetch your favorite Checkpoints (SDXL, SD 1.5, etc.), VAEs, and LoRAs.
3. **Launch ComfyUI:** Select your preferred tunneling method in the launch cell and run it. Click the generated link to open the ComfyUI interface!

---
*Template created and maintained by [Maybeyoou](https://github.com/maybeoff)*

In [ ]:
!git clone https://github.com/Comfy-Org/ComfyUI.git


In [ ]:
!cd ComfyUI && pip install -r requirements.txt && pip install blake3

Here you can choose how to open the site
1. cloudflare tunnel (not work in russia)
2. tuna.am (A Russian service for opening ports. Recommended for people from the CIS and Russia. Registration and authorization required.)
3. LocalTunel

In [ ]:
!curl -LO "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
!sudo chmod +x cloudflared-linux-amd64
print("test")
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8188 > cloud.txt 2>&1 &

!sleep 5
!until grep -q "trycloudflare.com" cloud.txt; do sleep 1; done
!grep -o "https://[^ ]*trycloudflare.com" cloud.txt | head -n 1

In [ ]:
#@markdown Вставьте ваш токен в поле и запустите ячейку.
TOKEN = "" #@param {type:"string"}

import os
import time
import re

if not TOKEN:
    raise ValueError("Пожалуйста, введите ваш токен Tuna в поле справа!")

print("1. Скачивание и установка Tuna...")
# Скачиваем скриптом
!curl -sSLf https://get.tuna.am | sh

# Проверяем, куда встал бинарник. Если он в текущей папке, переносим в /usr/local/bin для глобального доступа
if os.path.exists('./tuna'):
    !mv ./tuna /usr/local/bin/tuna
    !chmod +x /usr/local/bin/tuna

print("2. Сохранение токена...")
!tuna config save-token {TOKEN}

print("3. Запуск туннеля на порт 8188...")
# В Colab для фонового запуска без зависания ячейки лучше использовать system_raw
get_ipython().system_raw('nohup tuna http 8188 > tuna.log 2>&1 &')

print("Ожидание генерации ссылки...")
time.sleep(4)

# 4. Читаем логи и вытаскиваем ссылку
if os.path.exists('tuna.log'):
    with open('tuna.log', 'r') as f:
        log_content = f.read()

    # Ищем регуляркой ссылку вида https://*.tuna.am
    match = re.search(r'https://[a-zA-Z0-9.-]+.tuna.am', log_content)

    if match:
        print("\n" + "="*50)
        print(f" ТУННЕЛЬ УСПЕШНО ЗАПУЩЕН!")
        print(f" Ваша ссылка: {match.group(0)}")
        print("="*50)
    else:
        print("\n Ссылка не найдена в логах. Вот содержимое tuna.log:")
        print("-" * 40)
        print(log_content)
        print("-" * 40)
else:
    print(" Ошибка: файл логов tuna.log не был создан.")

In [ ]:
!npm install -g localtunnel
!lt --port 8188

In [ ]:
!cd ComfyUI && python main.py --listen 0.0.0.0 --enable-cors-header --enable-manager

In [ ]:
%%bash
cd ComfyUI

# Запуск с флагом -u для мгновенной записи в лог
nohup python -u main.py --listen 0.0.0.0 --enable-cors-header --enable-manager > log.txt 2>&1 &

# Даем секунду на старт процесса
sleep 1

echo "Waiting for ComfyUI to start..."
while ! grep -q "To see the GUI go to: http://0.0.0.0:8188" log.txt; do
    sleep 2
done

echo -e "\n🚀 ComfyUI run!"